In [27]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from sqlalchemy import and_, case, distinct, func, literal, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # import the needed tables for the extractions
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS224_APPLN_CPC,
    TLS230_APPLN_TECHN_FIELD,
    TLS901_TECHN_FIELD_IPC,
)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

In [28]:
# ---------------------------------------------------------------- environment
ENV = "TEST"                 # develop on TEST, switch to "PROD" for the final numbers
OUTPUT_DIR = Path("outputs/test/step0")

In [29]:
def symbol_prefix_condition(symbol_column, codes):   
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def keyword_condition(terms):   
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))
    

In [30]:
SECTOR_NAME = "Quantum Computing"
SECTOR_CPC_PREFIXES = (
    "G06N10/00",  # Calcoli o elaborazioni basati su modelli di architettura quantistica (inclusi computer quantistici, qubit)
    "G06N10/20",  # Architetture di computer quantistici
    "G06N10/40",  # Attuazione o manipolazione fisica dei qubit
    "G06N10/60",  # Algoritmi quantistici
    "B82Y 10/00", # Nanotecnologie per calcolo quantistico / elettronica molecolare
    "H10N 60/",   # Dispositivi superconduttori o quantistici (a seconda della versione PATSTAT)
)

SECTOR_KEYWORDS = (
    "quantum computer",
    "quantum computing",
    "qubit",
    "quantum processor",
    "quantum algorithm",
    "superconducting qubit",
    "quantum error correction",
)

In [31]:
IPR_TYPE = "PI"  # "PI" = Patented Inventions (esclude i modelli di utilità)

Family_id

In [32]:
def sector_family_ids(cpc_prefixes=SECTOR_CPC_PREFIXES, keywords=SECTOR_KEYWORDS):
    """Query dei docdb_family_id che appartengono al settore (codici CPC OR keyword)."""
    family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")
    by_cpc = (
        db.query(family_id)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, cpc_prefixes))
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE) 
    )
    if not keywords:
        return by_cpc.distinct()
    by_keywords = (
        db.query(family_id)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(keywords))
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE) 
    )
    return by_cpc.union(by_keywords)

# Eseguiamo la selezione della query
sector_ids = sector_family_ids()

Aplln_id

In [33]:
# Funzione per estrarre tutti i singoli brevetti (appln_id) delle famiglie del settore
def sector_application_ids(cpc_prefixes=SECTOR_CPC_PREFIXES, keywords=SECTOR_KEYWORDS):
    """Query di tutti gli appln_id appartenenti alle famiglie del settore."""
    
    # 1. Otteniamo prima gli ID di famiglia tramite la tua query
    fam_ids_subquery = sector_family_ids(cpc_prefixes, keywords).subquery().select()
    
    # 2. Selezioniamo tutte le applicazioni (tls201_appln) che appartengono a queste famiglie
    appln_query = (
        db.query(TLS201_APPLN.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(fam_ids_subquery))
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE) # Esclude i modelli di utilità se configurato come "PI"
    )
    
    return appln_query

# Eseguiamo la query per ottenere la lista di tutte le applicazioni
sector_apps = sector_application_ids()

In [34]:
# 1. Calcoliamo il numero totale di famiglie uniche trovate
total_families = db.query(sector_ids.subquery()).count()

# 2. Calcoliamo il numero totale di singoli brevetti (applicazioni) trovati
total_applications = db.query(sector_apps.subquery()).count()

print(f"--- RISULTATI DELLA QUERY (Ambiente: {ENV}) ---")
print(f"Settore: {SECTOR_NAME}")
print(f"Totale Famiglie di brevetti trovate (invenzioni uniche): {total_families:,}")
print(f"Totale Applicazioni trovate (singoli depositi globali): {total_applications:,}")

--- RISULTATI DELLA QUERY (Ambiente: TEST) ---
Settore: Quantum Computing
Totale Famiglie di brevetti trovate (invenzioni uniche): 66
Totale Applicazioni trovate (singoli depositi globali): 92
